<a href="https://colab.research.google.com/github/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02/blob/fusion_gnn/colab/kiem_chung_thay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kiểm chứng theo checklist của thầy — sau báo cáo Fusion GAT

Notebook này trả lời **13 đầu mục** thầy yêu cầu, chia 3 nhóm:

- **A. Dữ liệu ảnh** — nguồn/lô ảnh, trùng lặp với thư mục cũ, shortcut qua metadata, đơn vị chia fold
- **B. Phương pháp & rò rỉ** — embedding, điểm hàng xóm, node test, burn-in, định nghĩa nhãn và biến
- **C. Thí nghiệm bổ sung** — L0, so sánh ghép cặp, ablation B3b, calibration

Mỗi đầu mục có: câu hỏi nguyên văn → code → số đo → dòng **Kết luận** sinh tự động từ số đo.
Cell cuối cùng in bảng tổng hợp 13 mục. Phụ lục in mã nguồn các hàm tái sử dụng.

**Cách chạy trên Colab:** Runtime → Change runtime type → **T4 GPU**, rồi Runtime → **Run all**.
Trước đó phải tải file `kiemchung_data.zip` lên thư mục gốc Google Drive (xem cell cấu hình).

**Quyền riêng tư:** notebook **không in** tên file ảnh, họ tên hay số điện thoại. Chỉ in `img_id`
và `patient_uid` (mã băm 16 ký tự).

## 0. Cấu hình và nạp dữ liệu

In [1]:
# ============================== CAU HINH ==============================
RUN_ON_COLAB = True                     # False = chay local tu thu muc repo
REPO_URL = "https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git"
BRANCH = "fusion_gnn"
# File zip tai len thu muc goc Google Drive, chua dung cau truc:
#   data/train/{normal,silicosis}/*.jpg   data/test/{normal,silicosis}/*.jpg
#   data/sillicosis/*.jpg (thu muc cu, 535 anh)
#   data/info.csv   data/Main_data_Merged_1835_Images.xlsx
DRIVE_ZIP = "/content/drive/MyDrive/kiemchung_data.zip"

N_SEEDS, EPOCHS = 3, 60                 # giong het Phase 3
GAT_SEEDS, GAT_EPOCHS = (3, 300) if RUN_ON_COLAB else (1, 40)   # local chi chay thu nhanh
N_BOOT = 2000

In [2]:
import os, sys, shutil, subprocess, hashlib, inspect, json, warnings, tempfile
from pathlib import Path
warnings.filterwarnings("ignore")

if RUN_ON_COLAB:
    os.chdir("/content")
    shutil.rmtree("/content/repo", ignore_errors=True)
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    from google.colab import drive
    drive.mount("/content/drive")
    assert os.path.exists(DRIVE_ZIP), f"Khong thay {DRIVE_ZIP} tren Drive"
    subprocess.run(["unzip", "-q", "-o", DRIVE_ZIP, "-d", "/content/repo"], check=True)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
need = ["data/train", "data/test", "data/sillicosis", "data/info.csv",
        "data/Main_data_Merged_1835_Images.xlsx",
        "fusion/output/fusion_dataset_ketqua_control.npz", "fusion/output/tabular_ketqua.npz",
        "fusion/output/graph_ketqua_k10_bridge.npz", "output/ketqua/ts_predictions.parquet",
        "output/ketqua/fusion_node_meta.parquet", "imagefeat/output/image_index.parquet"]
missing = [f for f in need if not (ROOT / f).exists()]
assert not missing, f"THIEU: {missing}"
print("Thu muc lam viec:", ROOT, "| du file dau vao")

Mounted at /content/drive
Thu muc lam viec: /content/repo | du file dau vao


In [3]:
import numpy as np, pandas as pd, torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# Ham tai su dung — cung dung ham da sinh ra so lieu Phase 3/4 (ma nguon in o Phu luc)
from fusion.baselines import (standardise, neighbour_score_mean, image_probe_scores,
                              train_predict, score_block, paired_gap)
from imagefeat.build_index import extract_id, extract_batch_date

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
KL = {}   # ket luan tung dau muc -> in o cuoi

def ci(r):   # dinh dang "0.651 [0.625–0.677]"
    return f"{r['roc']:.3f} [{r['roc_ci'][0]:.3f}–{r['roc_ci'][1]:.3f}]"

def pgap(a, b, mask, n=N_BOOT):
    g, lo, hi, p = paired_gap(y[mask], a[mask], b[mask], uid[mask], n=n)
    return g, lo, hi, p

# ---- du lieu fusion (da commit, khong PII) ----
D = np.load(ROOT / "fusion/output/fusion_dataset_ketqua_control.npz", allow_pickle=False)
TAB = np.load(ROOT / "fusion/output/tabular_ketqua.npz", allow_pickle=False)
TS = pd.read_parquet(ROOT / "output/ketqua/ts_predictions.parquet")
G = np.load(ROOT / "fusion/output/graph_ketqua_k10_bridge.npz", allow_pickle=False)
assert (TAB["id"] == D["id"]).all() and (TS["id"].to_numpy() == D["id"]).all()

y = D["y"].astype(np.float32); yi = y.astype(int)
has = D["has_image"].astype(bool); uid = D["patient_uid"]
dev = D["split"] == "dev"; fold = D["fold_id"]; n = len(y)
X_tab = TAB["X_tab"].astype(np.float32)
X_ls = np.hstack([X_tab, TS["score"].to_numpy(np.float32)[:, None]])
X_img = np.nan_to_num(D["X_img"].astype(np.float32))
E = G["edge_index"]
mi, mu = dev & has, dev & ~has
print(f"{n} node | dev {dev.sum()} ({int(y[dev].sum())} duong) | test {(~dev).sum()} | "
      f"dev co anh {mi.sum()} ({int(y[mi].sum())}) | dev khong anh {mu.sum()} ({int(y[mu].sum())})")

8030 node | dev 6882 (823 duong) | test 1148 | dev co anh 1571 (393) | dev khong anh 5311 (430)


---
# A. DỮ LIỆU ẢNH

## A1. 1.835 ảnh lấy từ những lô nào; bảng chéo nguồn/lô ảnh × ketqua × tỉnh × năm

*"Nguồn"* = cách ảnh được lưu trong archive gốc: tách `train/test` và thư mục `normal/silicosis`.
*"Lô"* = ngày chụp rút từ tên file (`YYYYMMDD`). Tỉnh và năm lấy từ `info.csv` qua khoá `id`.

In [4]:
recs = []
for split in ("train", "test"):
    for cls in ("normal", "silicosis"):
        for p in sorted((DATA / split / cls).glob("*")):
            if p.is_file():
                recs.append({"img_id": extract_id(p.name), "nguon": split, "thu_muc": cls,
                             "lo": extract_batch_date(p.name), "path": p})
F = pd.DataFrame(recs)
assert len(F) == 1835 and F["img_id"].notna().all() and F["img_id"].is_unique, "Quet anh sai"
F["img_id"] = F["img_id"].astype(int)

INFO = pd.read_csv(DATA / "info.csv", usecols=["id", "tinh", "namdt", "ketqua", "bnn", "bnncuthe", "nam"])
A = F.merge(INFO, left_on="img_id", right_on="id", how="left")
assert A["ketqua"].notna().all(), "Co anh khong ghep duoc voi info.csv"
A["tinh"] = A["tinh"].str.strip().str.title()
A["duong"] = (A["ketqua"] == 1).astype(int)

khop_tm = ((A["thu_muc"] == "silicosis") == (A["duong"] == 1)).mean()
nam_lo = A["lo"].str[:4]
lech_nam = int(((nam_lo != "unkn") & (nam_lo != A["namdt"].astype(str))).sum())
print(f"Quet: {len(A)} anh, {A['lo'].nunique()} lo | thu muc normal/silicosis khop ketqua: {khop_tm:.1%}"
      f" | lo lech nam kham (namdt): {lech_nam} anh\n")

print("=== Nguon (train/test cua archive) x ketqua ===")
print(pd.crosstab(A["nguon"], A["ketqua"].map({1: "duong", 2: "am"}), margins=True))

print("\n=== Lo x ketqua (kem ty le duong) ===")
t = pd.crosstab(A["lo"], A["ketqua"].map({1: "duong", 2: "am"}), margins=True)
t["ty_le_duong"] = (t.get("duong", 0) / t["All"]).round(3)
print(t)

print("\n=== Lo x tinh ===");  print(pd.crosstab(A["lo"], A["tinh"], margins=True))
print("\n=== Lo x nam kham ==="); print(pd.crosstab(A["lo"], A["namdt"], margins=True))
print("\n=== Tinh x nam x ketqua (ty le duong) ===")
print(A.pivot_table(index="tinh", columns="namdt", values="duong", aggfunc=["count", "mean"]).round(3))

per_lo = A[A["lo"] != "unknown"].groupby("lo")["duong"].agg(["count", "mean"])
tinh_moi_lo = A[A["lo"] != "unknown"].groupby("lo")["tinh"].nunique()
cuc = per_lo[(per_lo["mean"] == 0) | (per_lo["mean"] == 1)]
KL["A1"] = (f"{A['lo'].nunique()} lo (ke ca 'unknown'); ty le duong theo lo tu {per_lo['mean'].min():.1%} "
            f"den {per_lo['mean'].max():.1%}; {len(cuc)} lo toan am/toan duong; "
            f"{(tinh_moi_lo == 1).sum()}/{len(tinh_moi_lo)} lo chi thuoc 1 tinh; "
            f"thu muc normal/silicosis khop ketqua {khop_tm:.1%}; {lech_nam} anh co nam trong ten file khac nam kham")
print("\nKET LUAN A1:", KL["A1"])

Quet: 1835 anh, 18 lo | thu muc normal/silicosis khop ketqua: 100.0% | lo lech nam kham (namdt): 207 anh

=== Nguon (train/test cua archive) x ketqua ===
ketqua    am  duong   All
nguon                    
test     269     91   360
train   1104    371  1475
All     1373    462  1835

=== Lo x ketqua (kem ty le duong) ===
ketqua      am  duong   All  ty_le_duong
lo                                      
20181219   171     28   199        0.141
20181222    76     18    94        0.191
20181223    91     74   165        0.448
20181224    31     44    75        0.587
20181227   139     17   156        0.109
20190103    31     14    45        0.311
20190114   129     35   164        0.213
20190115   144     38   182        0.209
20190605    75     54   129        0.419
20190606    74     44   118        0.373
20190607    18     13    31        0.419
20190620    78     23   101        0.228
20190621    54     26    80        0.325
20190827     0      6     6        1.000
20191113    69      4

## A2. Trong số ảnh dương tính, bao nhiêu ảnh trùng với thư mục sillicosis cũ (535 ảnh)?

So khớp hai tầng: (1) **SHA-256 nội dung file** — trùng tuyệt đối; (2) **điểm ảnh**: thu nhỏ về
64×64 xám, chuẩn hoá, lấy tương quan Pearson với ảnh giống nhất bên kia — bắt cả ảnh bị lưu lại,
đổi tên, nén lại. In cả phân bố tương quan để thầy tự thấy ngưỡng.

In [5]:
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def pix(p):
    a = np.asarray(Image.open(p).convert("L").resize((64, 64), Image.BILINEAR), dtype=np.float32).ravel()
    return (a - a.mean()) / (a.std() + 1e-6)

OLD = sorted(p for p in (DATA / "sillicosis").glob("*") if p.is_file())
print(f"Thu muc cu: {len(OLD)} file")
A["sha"] = A["path"].map(sha)
old_sha = [sha(p) for p in OLD]
P_new = np.stack([pix(p) for p in A["path"]])
P_old = np.stack([pix(p) for p in OLD])
corr = (P_new @ P_old.T) / P_new.shape[1]            # tuong quan Pearson
A["corr_max"] = corr.max(1)
A["trung_sha"] = A["sha"].isin(set(old_sha))
THR = 0.99
A["trung_pix"] = A["corr_max"] >= THR

for nhom, m in (("DUONG (ketqua=1)", A["duong"] == 1), ("AM (ketqua=2)", A["duong"] == 0)):
    print(f"{nhom:17s} n={m.sum():4d} | trung SHA: {A.loc[m, 'trung_sha'].sum():4d} | "
          f"trung diem anh (corr>={THR}): {A.loc[m, 'trung_pix'].sum():4d}")
old_in_new_sha = sum(s in set(A["sha"]) for s in old_sha)
old_in_new_pix = int((corr.max(0) >= THR).sum())
print(f"Anh cu co mat trong 1835: SHA {old_in_new_sha}/{len(OLD)} | diem anh {old_in_new_pix}/{len(OLD)}")
print("\nPhan bo tuong quan lon nhat cua moi anh moi voi thu muc cu:")
print(A.groupby("duong")["corr_max"].describe(percentiles=[.05, .5, .95]).round(4))

np_ = int((A["duong"] == 1).sum())
KL["A2"] = (f"{int(A.loc[A['duong']==1,'trung_pix'].sum())}/{np_} anh duong trung thu muc cu theo diem anh "
            f"({int(A.loc[A['duong']==1,'trung_sha'].sum())} trung SHA); "
            f"{int(A.loc[A['duong']==0,'trung_pix'].sum())} anh am trung; "
            f"{len(OLD) - old_in_new_pix}/{len(OLD)} anh cu KHONG co trong 1835")
print("\nKET LUAN A2:", KL["A2"])

Thu muc cu: 535 file
DUONG (ketqua=1)  n= 462 | trung SHA:  462 | trung diem anh (corr>=0.99):  462
AM (ketqua=2)     n=1373 | trung SHA:    0 | trung diem anh (corr>=0.99):    0
Anh cu co mat trong 1835: SHA 462/535 | diem anh 462/535

Phan bo tuong quan lon nhat cua moi anh moi voi thu muc cu:
        count    mean     std     min      5%     50%     95%     max
duong                                                                
0      1373.0  0.9198  0.0294  0.7024  0.8692  0.9253  0.9549  0.9682
1       462.0  1.0000  0.0000  1.0000  1.0000  1.0000  1.0000  1.0000

KET LUAN A2: 462/462 anh duong trung thu muc cu theo diem anh (462 trung SHA); 0 anh am trung; 73/535 anh cu KHONG co trong 1835


## A3. Bộ phân loại chỉ dùng metadata ảnh để dự đoán ketqua — báo cáo AUC

Đặc trưng: width, height, số byte, mode, bit depth, định dạng, nguồn (`train/test`), lô.
CV 5 fold **nhóm theo bệnh nhân**, hồi quy logistic + GBM, KTC 95% bootstrap.

**Hai biến thể:** (V1) không có tên thư mục `normal/silicosis`; (V2) có tên thư mục. V2 chỉ để
minh hoạ: tên thư mục **chính là nhãn** (khớp 100% ở A1), nên AUC ≈ 1 là tầm thường, không phải phát hiện.
AUC của V1 mới là câu trả lời: nếu cao, mô hình ảnh có thể học được lối tắt từ đặc tính file thay vì
nội dung phim.

In [6]:
BITS = {"1": 1, "L": 8, "P": 8, "RGB": 24, "RGBA": 32, "CMYK": 32, "I;16": 16, "I": 32, "F": 32}
meta = []
for p in A["path"]:
    with Image.open(p) as im:
        meta.append({"width": im.width, "height": im.height, "bytes": Path(p).stat().st_size,
                     "mode": im.mode, "bits": getattr(im, "bits", BITS.get(im.mode, -1)),
                     "format": im.format})
M = pd.concat([A[["img_id", "nguon", "thu_muc", "lo", "duong"]].reset_index(drop=True),
               pd.DataFrame(meta)], axis=1)
idx = pd.read_parquet(ROOT / "imagefeat/output/image_index.parquet")[["img_id", "patient_uid"]]
M = M.merge(idx, on="img_id", how="left"); assert M["patient_uid"].notna().all()
print("Gia tri metadata:")
for c in ("mode", "bits", "format"):
    print(f"  {c:7s}", M[c].value_counts().to_dict())
print("  width  ", M["width"].describe()[["min", "50%", "max"]].to_dict())
print("  height ", M["height"].describe()[["min", "50%", "max"]].to_dict())
print("  bytes  ", M["bytes"].describe()[["min", "50%", "max"]].to_dict())

def design(cols):
    num = [c for c in cols if c in ("width", "height", "bytes", "bits")]
    cat = [c for c in cols if c not in num]
    X = M[num].astype(float)
    if cat:
        X = pd.concat([X, pd.get_dummies(M[cat].astype(str))], axis=1)
    return X.to_numpy(float)

yA, gA = M["duong"].to_numpy(), M["patient_uid"].to_numpy()
def cv_auc(X, model):
    s = np.zeros(len(yA))
    for tr, va in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, yA, gA):
        m = (make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced"))
             if model == "lr" else HistGradientBoostingClassifier(random_state=0))
        s[va] = m.fit(X[tr], yA[tr]).predict_proba(X[va])[:, 1]
    return s

def boot(yv, s, gv):
    r = score_block(yv, s, gv); return r

V1 = ["width", "height", "bytes", "bits", "mode", "format", "nguon", "lo"]
V2 = V1 + ["thu_muc"]
res_A3 = {}
for ten, cols in (("V1 — khong ten thu muc", V1), ("V2 — CO ten thu muc (minh hoa)", V2)):
    for mdl in ("lr", "gbm"):
        r = score_block(yA, cv_auc(design(cols), mdl), gA)
        res_A3[(ten, mdl)] = r
        print(f"{ten:32s} {mdl:3s}  ROC-AUC {ci(r)}")

print("\nAUC theo tung nhom dac trung (hoi quy logistic, cung CV):")
nhom = {"kich thuoc (width,height)": ["width", "height"], "so byte": ["bytes"],
        "mode+bits+dinh dang": ["mode", "bits", "format"], "nguon train/test": ["nguon"], "lo chup": ["lo"]}
auc_nhom = {}
for k_, cols in nhom.items():
    auc_nhom[k_] = roc_auc_score(yA, cv_auc(design(cols), "lr"))
    print(f"  {k_:28s} {auc_nhom[k_]:.3f}")
best = max(res_A3[("V1 — khong ten thu muc", "lr")]["roc"], res_A3[("V1 — khong ten thu muc", "gbm")]["roc"])
top = max(auc_nhom, key=auc_nhom.get)
KL["A3"] = (f"Metadata KHONG ten thu muc: ROC-AUC tot nhat {best:.3f} (LR {ci(res_A3[('V1 — khong ten thu muc','lr')])}, "
            f"GBM {ci(res_A3[('V1 — khong ten thu muc','gbm')])}); nhom manh nhat: {top} ({auc_nhom[top]:.3f}). "
            f"Co ten thu muc: {res_A3[('V2 — CO ten thu muc (minh hoa)','lr')]['roc']:.3f} (tam thuong vi thu muc = nhan)")
print("\nKET LUAN A3:", KL["A3"])

Gia tri metadata:
  mode    {'L': 1828, 'RGB': 7}
  bits    {8: 1835}
  format  {'JPEG': 1835}
  width   {'min': 2452.0, '50%': 3068.0, 'max': 3072.0}
  height  {'min': 2001.0, '50%': 3060.0, 'max': 3072.0}
  bytes   {'min': 1113454.0, '50%': 1891106.0, 'max': 4936694.0}
V1 — khong ten thu muc           lr   ROC-AUC 0.698 [0.670–0.725]
V1 — khong ten thu muc           gbm  ROC-AUC 0.674 [0.646–0.702]
V2 — CO ten thu muc (minh hoa)   lr   ROC-AUC 1.000 [1.000–1.000]
V2 — CO ten thu muc (minh hoa)   gbm  ROC-AUC 1.000 [1.000–1.000]

AUC theo tung nhom dac trung (hoi quy logistic, cung CV):
  kich thuoc (width,height)    0.536
  so byte                      0.522
  mode+bits+dinh dang          0.502
  nguon train/test             0.467
  lo chup                      0.704

KET LUAN A3: Metadata KHONG ten thu muc: ROC-AUC tot nhat 0.698 (LR 0.698 [0.670–0.725], GBM 0.674 [0.646–0.702]); nhom manh nhat: lo chup (0.704). Co ten thu muc: 1.000 (tam thuong vi thu muc = nhan)


## A4. Đơn vị gom nhóm khi chia fold là lượt khám hay người, và cách xác định cùng một người

Trả lời: gom theo **người** (`patient_uid`). Luật định danh trích nguyên văn từ
`scripts/patient_identity.py` bên dưới; các assert chạy trên dữ liệu thật.

In [7]:
import scripts.patient_identity as PI
doc = PI.__doc__
print(doc[doc.index("LUAT XAC DINH"):doc.index("HAI COT DAU RA")].rstrip())

META = pd.read_parquet(ROOT / "output/ketqua/fusion_node_meta.parquet")
assert (META["id"].to_numpy() == D["id"]).all()
luot = len(META); nguoi = META["patient_uid"].nunique()
vc = META["patient_uid"].value_counts()
print(f"\nToan bo: {luot} luot kham / {nguoi} nguoi | {int((vc > 1).sum())} nguoi kham nhieu lan, "
      f"toi da {vc.max()} lan")
img_uid = uid[has]
print(f"Nhom co anh: {has.sum()} luot / {len(set(img_uid))} nguoi")

dv = META[META["split"] == "dev"]
vat_fold = int((dv.groupby("patient_uid")["fold_id"].nunique() > 1).sum())
vat_split = len(set(META.loc[META.split == "dev", "patient_uid"]) & set(META.loc[META.split == "test", "patient_uid"]))
assert vat_fold == 0 and vat_split == 0
print(f"Kiem tra tren du lieu: nguoi nam o >1 fold = {vat_fold} | nguoi vat dev/test = {vat_split}")
print("Cach chia: StratifiedGroupKFold(groups=patient_uid): 1/7 lam test, dev chia 5 fold (DO_AN.ipynb cell 8)")
KL["A4"] = (f"Gom theo NGUOI (patient_uid), khong theo luot: {luot} luot = {nguoi} nguoi; "
            f"0 nguoi vat fold, 0 nguoi vat dev/test; cung mot nguoi = trung SDT, hoac thieu SDT thi trung ca 6 truong")
print("\nKET LUAN A4:", KL["A4"])

LUAT XAC DINH "CUNG MOT NGUOI":
  1. Trung SO DIEN THOAI                      -> cung mot nguoi.
  2. Neu it nhat mot ben THIEU so dien thoai  -> phai trung CA 6 truong:
     ho ten, nam sinh, tinh, gioi tinh, cap do nghe `cviec`, cap do nghe `pxuong`.
  3. Con lai                                  -> hai nguoi KHAC NHAU.

Cap do nghe dung dung ham classify_job_10_levels() ben duoi, nen
"Van hanh may" va "van hanh may moc" quy ve cung mot cap.

Toan bo: 8030 luot kham / 7024 nguoi | 1002 nguoi kham nhieu lan, toi da 4 lan
Nhom co anh: 1835 luot / 1748 nguoi
Kiem tra tren du lieu: nguoi nam o >1 fold = 0 | nguoi vat dev/test = 0
Cach chia: StratifiedGroupKFold(groups=patient_uid): 1/7 lam test, dev chia 5 fold (DO_AN.ipynb cell 8)

KET LUAN A4: Gom theo NGUOI (patient_uid), khong theo luot: 8030 luot = 7024 nguoi; 0 nguoi vat fold, 0 nguoi vat dev/test; cung mot nguoi = trung SDT, hoac thieu SDT thi trung ca 6 truong


---
# B. PHƯƠNG PHÁP VÀ RÒ RỈ

## B5. Embedding "control" và "frozen": encoder gì, input bao nhiêu, có fine-tune không, trên dữ liệu nào, có out-of-fold không?

In [8]:
for ten in ("control", "frozen"):
    mf = json.loads((ROOT / f"imagefeat/output/image_features_{ten}_manifest.json").read_text(encoding="utf-8"))
    print(f"--- {ten} ---")
    for k in ("role", "checkpoint", "backbone", "train_label_of_checkpoint", "leakage_claim",
              "transform", "feature_dim_raw", "feature_dim_kept", "samples"):
        if k in mf: print(f"  {k:26s}: {mf[k]}")

src = (ROOT / "Finetune CrossEntropy/finetune_biovilt_ce.py").read_text(encoding="utf-8").splitlines()
print("\nScript fine-tune cua checkpoint control (Finetune CrossEntropy/finetune_biovilt_ce.py):")
for i, l in enumerate(src, 1):
    if any(k in l for k in ("EXCEL_PATH", "LABEL_COLUMN", "RESIZE ", "CROP ", "GroupShuffleSplit(", "EPOCHS ")):
        print(f"  dong {i}: {l.strip()}")

seta = pd.read_excel(ROOT / "Finetune 2/SetA_Labels.xlsx")
print(f"\nSetA: {len(seta)} anh | nhan bnn: {seta['bnn'].value_counts().to_dict()}")
stem = lambda s: Path(str(s)).stem.lower().strip()
seta_names = set(seta["file_name"].map(stem))
new_names = set(A["path"].map(lambda p: p.stem.lower().strip()))
old_names = set(p.stem.lower().strip() for p in OLD)
print(f"Trung TEN file SetA <-> 1835 anh: {len(seta_names & new_names)} | SetA <-> thu muc cu 535: {len(seta_names & old_names)}")
KL["B5"] = ("Ca hai: BioViL-T ResNet50, anh xam Resize 512 -> CenterCrop 448 -> 3 kenh, lay 256/512 chieu song. "
            "control = fine-tune tren SetA (bo du lieu NGOAI, 2129 anh, nhan bnn), MOT mo hinh ap cho ca 1835 anh "
            "-> KHONG out-of-fold va KHONG hoc tren du lieu benh vien. frozen = khong fine-tune. "
            f"Trung ten file SetA-1835: {len(seta_names & new_names)}; chua so duoc hash vi khong co anh goc SetA")
print("\nKET LUAN B5:", KL["B5"])

--- control ---
  role                      : BO DOI CHUNG — khong cross-fit, dung de kiem chung ket qua fusion
  checkpoint                : C:\Users\ASUS\AndroidStudioProjects\cgv1\New folder (2)\DO-AN-TOT-NGHIEP-E22-T02\Finetune CrossEntropy\best_model_ce.pth
  train_label_of_checkpoint : bnn (nhanh SetA, GroupShuffleSplit seed 42)
  leakage_claim             : SetA va NEW DATA gia dinh khong giao nhau — CHUA kiem chung bang so
  transform                 : Resize(512) -> CenterCrop(448) -> ToTensor -> ExpandChannels
  feature_dim_raw           : 512
  feature_dim_kept          : 256
  samples                   : 1835
--- frozen ---
  role                      : BO FROZEN — backbone BioViL-T pretrained goc, KHONG fine-tune, dung de doi chung dac trung it collapse
  checkpoint                : None
  backbone                  : /content/DO-AN-TOT-NGHIEP-E22-T02/Pre-train BioViL-T/biovil_t_image_model_proj_size_128.pt
  train_label_of_checkpoint : None
  leakage_claim             : Kh

## B6. "Điểm nguy cơ hàng xóm" lấy từ mô hình nào; out-of-fold hay in-sample?

Mô hình: hồi quy logistic (C=0,1, cân bằng lớp) trên 256 chiều ảnh `control`, **fit lại trong từng
fold** chỉ trên node train có ảnh, rồi chấm cho mọi node có ảnh (`image_probe_scores`). Một node được
gán **trung bình điểm của các hàng xóm có ảnh**. Ô dưới đo chính xác mỗi điểm hàng xóm là in-sample
hay out-of-fold, tính trên các node validation của 5 fold.

In [9]:
src_e = np.concatenate([E[0], E[1]]); dst_e = np.concatenate([E[1], E[0]])   # doi xung
dem = {"in-sample (hang xom thuoc fold TRAIN)": 0, "out-of-fold (hang xom cung fold VALIDATION)": 0,
       "out-of-fold (hang xom thuoc TEST)": 0}
dem_mu = dict.fromkeys(dem, 0)
for k in range(5):
    va = dev & (fold == k); tr = dev & (fold != k) & (fold >= 0)
    keep = va[src_e] & has[dst_e]                 # canh tu node validation toi hang xom CO anh
    d_ = dst_e[keep]; s_ = src_e[keep]
    for ten, m in (("in-sample (hang xom thuoc fold TRAIN)", tr[d_]),
                   ("out-of-fold (hang xom cung fold VALIDATION)", va[d_]),
                   ("out-of-fold (hang xom thuoc TEST)", ~dev[d_])):
        dem[ten] += int(m.sum()); dem_mu[ten] += int((m & ~has[s_]).sum())
tong = sum(dem.values()); tong_mu = sum(dem_mu.values())
print("Nguon goc cac diem hang xom duoc dung cho node VALIDATION:")
for ten in dem:
    print(f"  {ten:46s} {dem[ten]:6d} ({dem[ten]/tong:.1%}) | rieng node khong anh: {dem_mu[ten]/tong_mu:.1%}")
print("\nNhan cua CHINH node dang cham: KHONG bao gio duoc dung (khong co self-loop; node khong anh "
      "khong he vao probe anh).")

# --- Vi sao gan nhu moi hang xom lai cung fold? Kiem tra cau truc do thi ---
fg = fold.copy(); fg[~dev] = 5                     # coi test la nhom thu 6
cung_fold = float((fg[E[0]] == fg[E[1]]).mean())
ky_vong = float(sum(np.mean(fg == k) ** 2 for k in range(6)))
from sklearn.model_selection import cross_val_predict, StratifiedKFold
pf = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), D["X_ts"], fg,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0))
doan_fold = float((pf == fg).mean())
print(f"\nKIEM TRA CAU TRUC DO THI:")
print(f"  Canh noi hai node CUNG fold: {cung_fold:.1%} | ky vong neu hang xom chon theo lam sang (ngau nhien voi fold): {ky_vong:.1%}")
print(f"  Chi nhin vector ts 32 chieu (X_ts) -> doan dung fold: {doan_fold:.1%} (ngau nhien ~{np.bincount(fg).max()/n:.1%})")
print("  Nguyen nhan: do thi kNN duoc dung tren X_ts. Tu khi trich vector ts OUT-OF-FOLD (node fold k lay tu model k,"
      "\n  node test lay trung binh 5 model), X_ts nam o 6 KHONG GIAN khac nhau -> kNN tim hang xom trong cung khong gian,"
      "\n  tuc CUNG FOLD, thay vi nguoi giong nhau ve lam sang. Moi ket qua dung do thi nay (B3, B3b, GAT, homophily 1,25x)"
      "\n  deu do tren mot do thi hong cau truc.")
KL["B6"] = (f"Hoi quy logistic tren anh control, fit lai moi fold tren node train co anh. Diem hang xom: "
            f"{dem['in-sample (hang xom thuoc fold TRAIN)']/tong:.1%} in-sample, con lai out-of-fold; nhan cua chinh "
            f"node khong duoc dung. PHAT HIEN: {cung_fold:.1%} canh noi node CUNG fold (ky vong {ky_vong:.1%}) vi do thi "
            f"dung tren X_ts ma X_ts doan duoc fold {doan_fold:.1%} -> do thi hien tai hong cau truc")
print("\nKET LUAN B6:", KL["B6"])

Nguon goc cac diem hang xom duoc dung cho node VALIDATION:
  in-sample (hang xom thuoc fold TRAIN)           15735 (67.7%) | rieng node khong anh: 67.8%
  out-of-fold (hang xom cung fold VALIDATION)      4090 (17.6%) | rieng node khong anh: 17.6%
  out-of-fold (hang xom thuoc TEST)                3403 (14.7%) | rieng node khong anh: 14.6%

Nhan cua CHINH node dang cham: KHONG bao gio duoc dung (khong co self-loop; node khong anh khong he vao probe anh).

KIEM TRA CAU TRUC DO THI:
  Canh noi hai node CUNG fold: 17.0% | ky vong neu hang xom chon theo lam sang (ngau nhien voi fold): 16.7%
  Chi nhin vector ts 32 chieu (X_ts) -> doan dung fold: 98.9% (ngau nhien ~17.1%)
  Nguyen nhan: do thi kNN duoc dung tren X_ts. Tu khi trich vector ts OUT-OF-FOLD (node fold k lay tu model k,
  node test lay trung binh 5 model), X_ts nam o 6 KHONG GIAN khac nhau -> kNN tim hang xom trong cung khong gian,
  tuc CUNG FOLD, thay vi nguoi giong nhau ve lam sang. Moi ket qua dung do thi nay (B3, B3b, GAT, ho

## B7. Node test (1.148) có nằm trong đồ thị không; nhãn test có đi vào điểm hàng xóm ở bất kỳ bước nào không?

Ba phần: (1) đếm node/cạnh test trong đồ thị; (2) **phép thử xáo nhãn**: xáo ngẫu nhiên nhãn 1.148 node
test rồi dựng lại đặc trưng hàng xóm + chạy lại L3b — dự đoán trên dev phải **giống hệt từng bit** nếu
nhãn test không đi vào đâu; (3) khai báo: việc chọn cấu hình đồ thị có dùng nhãn test không.

In [10]:
test = ~dev
e_test = int((test[E[0]] | test[E[1]]).sum())
print(f"(1) Node test trong do thi: {test.sum()}/{n} (do thi transductive tren ca 8030 node) | "
      f"canh cham node test: {e_test}/{E.shape[1]} ({e_test/E.shape[1]:.1%})")

def nb_feat(k, kind, ylab):
    tr = dev & (fold != k) & (fold >= 0)
    if kind == "nbs":
        s = image_probe_scores(tr & has, X_img, ylab, has); nb, c = neighbour_score_mean(E, s, has)
    elif kind == "nblab":
        nb, c = neighbour_score_mean(E, ylab.astype(float), has & tr)   # CHI nhan hang xom thuoc fold train
    elif kind == "nbclin":
        clin = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
        s = clin.fit(X_ls[tr], ylab[tr]).predict_proba(X_ls)[:, 1]; nb, c = neighbour_score_mean(E, s, has)
    return np.column_stack([nb, np.log1p(c)]).astype(np.float32), c > 0

RAW = {"ls": X_ls, "tab": X_tab, "img": X_img}
VALID = {"ls": np.ones(n, bool), "tab": np.ones(n, bool), "img": has}

def run(keys, maskable, kind, rows=dev, ylab=None, seeds=N_SEEDS, nb_fold=None):
    """OOF 5 fold. Dac trung hang xom tinh lai TRONG TUNG FOLD (nb_fold=int -> tai hien loi cu)."""
    ylab = y if ylab is None else ylab
    oof = np.zeros(n)
    for k in range(5):
        va = rows & (fold == k); tr = rows & (fold != k) & (fold >= 0)
        R, V = dict(RAW), dict(VALID)
        for key in keys:
            if key.startswith("nb"):
                R[key], V[key] = nb_feat(k if nb_fold is None else nb_fold, key, ylab)
        Z = [standardise(tr, R[key], V[key]) for key in keys]
        if kind == "lr":
            parts = []
            for key, m_, z in zip(keys, maskable, Z):
                parts.append(z)
                if m_: parts.append(V[key][:, None].astype(np.float32))
            X = np.hstack(parts)
            mdl = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
            oof[va] = mdl.fit(X[tr], ylab[tr]).predict_proba(X[va])[:, 1]
        else:
            preds = [train_predict([z[tr] for z in Z], [V[key][tr] for key in keys], ylab[tr],
                                   [z[va] for z in Z], [V[key][va] for key in keys], maskable, s, EPOCHS, 64)
                     for s in range(seeds)]
            oof[va] = np.mean(preds, axis=0)
    return oof

rng = np.random.default_rng(123)
y_x = y.copy(); ti = np.flatnonzero(test); y_x[ti] = rng.permutation(y[ti])
print(f"\n(2) Xao nhan test: {int((y_x[ti] != y[ti]).sum())}/{len(ti)} nhan test bi doi")
same_feat = all(np.array_equal(nb_feat(k, "nbs", y)[0], nb_feat(k, "nbs", y_x)[0]) for k in range(5))
o1 = run(["ls", "img", "nbs"], [False, True, True], "lr")
o2 = run(["ls", "img", "nbs"], [False, True, True], "lr", ylab=y_x)
lech = float(np.abs(o1[dev] - o2[dev]).max())
print(f"    Dac trung diem hang xom giong het nhau o ca 5 fold: {same_feat}")
print(f"    L3b: chenh lech lon nhat cua du doan dev sau khi xao nhan test = {lech:.2e}")

print("\n(3) Chon cau hinh do thi: homophily lop duong tinh tren CA nhan test (nhu Phase 2) va CHI tren dev")
for gname in ("graph_ketqua_k5", "graph_ketqua_k10_bridge", "graph_ketqua_k20_bridge"):
    Eg = np.load(ROOT / f"fusion/output/{gname}.npz")["edge_index"]
    s2 = np.concatenate([Eg[0], Eg[1]]); d2 = np.concatenate([Eg[1], Eg[0]])
    def lift(mask):
        k_ = mask[s2] & mask[d2] & (yi[s2] == 1)
        return (yi[d2][k_] == 1).mean() / yi[mask].mean()
    print(f"    {gname:26s} lift ca 8030 node {lift(np.ones(n, bool)):.3f}x | chi dev {lift(dev):.3f}x")
KL["B7"] = (f"Node test CO nam trong do thi ({test.sum()} node, {e_test/E.shape[1]:.1%} canh). "
            f"Nhan test KHONG di vao mo hinh: xao nhan test -> dac trung hang xom giong het ({same_feat}), "
            f"du doan dev lech toi da {lech:.1e}. Khai bao: chon k=10 dua tren homophily co tinh ca nhan test "
            f"(chi de chon cau hinh, khong vao dac trung)")
print("\nKET LUAN B7:", KL["B7"])

(1) Node test trong do thi: 1148/8030 (do thi transductive tren ca 8030 node) | canh cham node test: 15362/58236 (26.4%)

(2) Xao nhan test: 240/1148 nhan test bi doi
    Dac trung diem hang xom giong het nhau o ca 5 fold: True
    L3b: chenh lech lon nhat cua du doan dev sau khi xao nhan test = 0.00e+00

(3) Chon cau hinh do thi: homophily lop duong tinh tren CA nhan test (nhu Phase 2) va CHI tren dev
    graph_ketqua_k5            lift ca 8030 node 1.235x | chi dev 1.240x
    graph_ketqua_k10_bridge    lift ca 8030 node 1.251x | chi dev 1.186x
    graph_ketqua_k20_bridge    lift ca 8030 node 1.209x | chi dev 1.227x

KET LUAN B7: Node test CO nam trong do thi (1148 node, 26.4% canh). Nhan test KHONG di vao mo hinh: xao nhan test -> dac trung hang xom giong het (True), du doan dev lech toi da 0.0e+00. Khai bao: chon k=10 dua tren homophily co tinh ca nhan test (chi de chon cau hinh, khong vao dac trung)


## B8. Có xử lý chữ burn-in trên ảnh không?

In [11]:
ex = (ROOT / "imagefeat/extract_image_features.py").read_text(encoding="utf-8").splitlines()
print("Tien xu ly anh khi trich embedding (imagefeat/extract_image_features.py):")
for i, l in enumerate(ex, 1):
    if "RESIZE, CROP" in l or ("Compose(" in l and "Resize" in l) or ".convert(" in l:
        print(f"  dong {i}: {l.strip()}")
tu_khoa = ("burn", "ocr", "inpaint", "mask_text", "text_mask")
co = [f for f in ("imagefeat/extract_image_features.py", "Finetune CrossEntropy/finetune_biovilt_ce.py")
      if any(k in (ROOT / f).read_text(encoding="utf-8").lower() for k in tu_khoa)]
print(f"File co buoc xu ly chu burn-in: {co if co else 'KHONG CO'}")
KL["B8"] = ("KHONG. Anh chi qua Resize 512 -> CenterCrop 448 (crop giua co the cat mot phan chu o mep, nhung "
            "khong co buoc che/xoa chu nao). Muc do chu burn-in anh huong mo hinh CHUA duoc do")
print("\nKET LUAN B8:", KL["B8"])

Tien xu ly anh khi trich embedding (imagefeat/extract_image_features.py):
  dong 70: RESIZE, CROP = 512, 448
  dong 84: return self.transform(Image.open(self.paths[i]).convert("L"))
  dong 173: eval_tf = Compose([Resize(RESIZE), CenterCrop(CROP), ToTensor(), ExpandChannels()])
File co buoc xu ly chu burn-in: KHONG CO

KET LUAN B8: KHONG. Anh chi qua Resize 512 -> CenterCrop 448 (crop giua co the cat mot phan chu o mep, nhung khong co buoc che/xoa chu nao). Muc do chu burn-in anh huong mo hinh CHUA duoc do


## B9. Định nghĩa chính xác nhãn bnn, và "timeseries" gồm những biến nào / mấy thời điểm

Danh sách biến được **dựng lại bằng chính code** cell 3–8 của `ĐỒ_ÁN.ipynb` (không tự liệt kê tay).

In [12]:
import io, contextlib
nb = json.loads((ROOT / "ĐỒ_ÁN.ipynb").read_text(encoding="utf-8"))
ns = {"os": os, "np": np, "pd": pd, "TARGET": "bnn", "OUT_DIR": str(ROOT / "output")}
cwd = os.getcwd(); os.chdir(ROOT)
with contextlib.redirect_stdout(io.StringIO()):
    for i in (3, 4, 5, 6, 7, 8):
        exec("".join(nb["cells"][i]["source"]), ns)
os.chdir(cwd)
Xs, Xn, Xj = ns["X_seq_all"], ns["X_num_all_raw"], ns["X_job_all"]
print(f"Dau vao mo hinh nhanh 'timeseries': chuoi {Xs.shape} | so {Xn.shape} | nghe {Xj.shape}")
zones = {i: [c for c in ns["seq_cols_flat"] if c.endswith(str(i))] for i in range(1, 7)}
print("'Chuoi' LSTM = 6 BUOC, moi buoc la MOT VUNG PHOI, moi vung 7 dau hieu kham:")
for i, cols in zones.items():
    print(f"  vung {i}: {cols}")
print(f"Khoi so ({len(ns['num_cols'])} cot): {ns['num_cols']}")
print(f"Khoi nghe: {ns['JOB_COLS']} -> ma hoa 10 cap")

print("\n--- Nhan bnn ---")
print("Trong info.csv, bnn nam trong khoi tien su: tiensuhh, benhhh, tsnoi, tsngoai, bnn, bnncuthe, n9, nam, sobh")
print("Gia tri bnn (1835 nguoi co anh):", INFO["bnn"].value_counts().to_dict())
print("bnncuthe khi bnn=Co:", INFO.loc[INFO.bnn == "Co", "bnncuthe"].value_counts().to_dict())
b = INFO[INFO.bnn == "Co"]; dnam = (b["namdt"] - b["nam"]).dropna()
print(f"Nam chan doan BNN (cot nam) so voi nam dot kham (namdt): truoc {(dnam > 0).sum()} | cung nam {(dnam == 0).sum()} "
      f"| sau {(dnam < 0).sum()} | trung vi cach {dnam.median():.0f} nam")
print("Doi chieu bnn x ketqua (1835 nguoi):")
print(pd.crosstab(INFO["bnn"], INFO["ketqua"].map({1: "phim duong", 2: "phim am"})))
KL["B9"] = (f"bnn = cau hoi TIEN SU 'da duoc chan doan benh nghe nghiep' (Co/Khong), {(dnam > 0).sum()}/{len(dnam)} ca "
            f"chan doan truoc dot kham. 'Timeseries' KHONG phai chuoi thoi gian: CHI 1 THOI DIEM; LSTM chay "
            f"tren 6 vung phoi x 7 dau hieu, cong {Xn.shape[1]} bien so va 4 cot nghe")
print("\nKET LUAN B9:", KL["B9"])

Dau vao mo hinh nhanh 'timeseries': chuoi (8030, 6, 7) | so (8030, 92) | nghe (8030, 4)
'Chuoi' LSTM = 6 BUOC, moi buoc la MOT VUNG PHOI, moi vung 7 dau hieu kham:
  vung 1: ['rungt1', 'rungg1', 'goduc1', 'vtam1', 'vtno1', 'vtrit1', 'vtngay1']
  vung 2: ['rungt2', 'rungg2', 'goduc2', 'vtam2', 'vtno2', 'vtrit2', 'vtngay2']
  vung 3: ['rungt3', 'rungg3', 'goduc3', 'vtam3', 'vtno3', 'vtrit3', 'vtngay3']
  vung 4: ['rungt4', 'rungg4', 'goduc4', 'vtam4', 'vtno4', 'vtrit4', 'vtngay4']
  vung 5: ['rungt5', 'rungg5', 'goduc5', 'vtam5', 'vtno5', 'vtrit5', 'vtngay5']
  vung 6: ['rungt6', 'rungg6', 'goduc6', 'vtam6', 'vtno6', 'vtrit6', 'vtngay6']
Khoi so (92 cot): ['gioitinh', 'tuoinghe', 'nampx', 'cv5nam', 'tgian1', 'tgian2', 'cao', 'can', 'hatd', 'hatt', 'mach', 'theluc', 'hutthuoc', 'slthuoc', 'tsnoi', 'tsngoai', 'ho', 'tdho', 'tsho', 'n1', 'khacdom', 'loaidom', 'tdkhacdo', 'khotho', 'mdkhotho', 'tdkhotho', 'daunguc', 'vitridau', 'daulan', 'tcdau', 'n2', 'tgdau', 'ytodau', 'n3', 'ytotang', 'ch

---
# C. THÍ NGHIỆM BỔ SUNG

Cùng giao thức Phase 3: chỉ tập dev (6.882 node), 5 fold có sẵn (nhóm theo người), mạng nơ-ron chạy 3 seed
× 60 epoch, KTC 95% bootstrap theo bệnh nhân. Tập test **không dùng**.

**Sửa lỗi phát hiện khi làm notebook này:** trong `fusion/baselines.py`, các mô hình hồi quy logistic có khối
điểm hàng xóm (L3b) dựng đặc trưng hàng xóm trong vòng lặp fold nhưng chỉ huấn luyện **sau** vòng lặp, nên cả 5
fold đều dùng đặc trưng của fold cuối. Notebook này tính lại L3b **đúng** (dựng đặc trưng trong từng fold), và
in thêm bản tái hiện lỗi cũ để thấy chênh lệch. B3b (mạng nơ-ron) không bị lỗi này.

Ô dưới chạy toàn bộ mô hình một lần (mất vài phút), các đầu mục sau chỉ đọc kết quả.

In [14]:
import time
t0 = time.time()
OOF = {}
OOF["L0 (lam sang + diem ts)"] = run(["ls"], [False], "lr")
OOF["L0-tho (chi bang tho)"] = run(["tab"], [False], "lr")
OOF["L2"] = run(["ls", "img"], [False, True], "lr")
OOF["L3b"] = run(["ls", "img", "nbs"], [False, True, True], "lr")
OOF["L3b (tai hien loi cu)"] = run(["ls", "img", "nbs"], [False, True, True], "lr", nb_fold=4)
OOF["L3b-(i) nhan hang xom"] = run(["ls", "img", "nblab"], [False, True, True], "lr")
OOF["L3b-(ii) diem lam sang hang xom"] = run(["ls", "img", "nbclin"], [False, True, True], "lr")
print(f"Hoi quy logistic xong ({time.time()-t0:.0f}s)")
OOF["B0"] = run(["ls"], [False], "mlp")
OOF["B2"] = run(["ls", "img"], [False, True], "mlp")
OOF["B3b"] = run(["ls", "img", "nbs"], [False, True, True], "mlp")
OOF["B3b-(i) nhan hang xom"] = run(["ls", "img", "nblab"], [False, True, True], "mlp")
OOF["B3b-(ii) diem lam sang hang xom"] = run(["ls", "img", "nbclin"], [False, True, True], "mlp")
print(f"Mang no-ron xong ({time.time()-t0:.0f}s)")

tmp = Path(tempfile.mkdtemp())
r = subprocess.run([sys.executable, "fusion/gnn.py", "--source", "control", "--seeds", str(GAT_SEEDS),
                    "--epochs", str(GAT_EPOCHS), "--out-dir", str(tmp)], cwd=ROOT, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
OOF["GAT"] = np.load(tmp / "phase4_oof_gat_control.npz")["GAT"]
print(f"GAT xong ({GAT_SEEDS} seed x toi da {GAT_EPOCHS} epoch) ({time.time()-t0:.0f}s)")

def bang(names, groups=(("toan dev", dev), ("co anh", mi), ("khong anh", mu))):
    rows = []
    for nm in names:
        row = {"mo hinh": nm}
        for g_, m in groups:
            rr = score_block(yi[m], OOF[nm][m], uid[m]); row[f"ROC {g_}"] = ci(rr); row[f"PR {g_}"] = f"{rr['pr']:.3f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("mo hinh")
print("\nDoi chieu tai lap voi bao cao Phase 3 (nhom khong anh):")
for nm, cu in (("L2", 0.651), ("L3b (tai hien loi cu)", 0.648), ("B3b", 0.640)):
    moi = roc_auc_score(yi[mu], OOF[nm][mu])
    print(f"  {nm:24s} bao cao {cu:.3f} | lan nay {moi:.4f} | {'KHOP' if abs(moi-cu) <= 0.005 else 'LECH ' + f'{moi-cu:+.4f}'}")
print("  (Hoi quy logistic la tat dinh nen phai khop; mang no-ron co the lech do seed/thiet bi GPU vs CPU)")

Hoi quy logistic xong (40s)
Mang no-ron xong (309s)
GAT xong (3 seed x toi da 300 epoch) (356s)

Doi chieu tai lap voi bao cao Phase 3 (nhom khong anh):
  L2                       bao cao 0.651 | lan nay 0.6509 | KHOP
  L3b (tai hien loi cu)    bao cao 0.648 | lan nay 0.6529 | KHOP
  B3b                      bao cao 0.640 | lan nay 0.6329 | LECH -0.0071
  (Hoi quy logistic la tat dinh nen phai khop; mang no-ron co the lech do seed/thiet bi GPU vs CPU)


## C10. L0: hồi quy logistic chỉ dùng lâm sàng, báo cáo riêng nhóm có ảnh / không ảnh

Hai biến thể: **L0** = bảng thô 139 cột + điểm dự đoán nhánh ts (đúng đầu vào của B0); **L0-thô** = chỉ bảng thô.

In [15]:
t10 = bang(["L0 (lam sang + diem ts)", "L0-tho (chi bang tho)", "L2"])
print(t10)
KL["C10"] = (f"L0: co anh {ci(score_block(yi[mi], OOF['L0 (lam sang + diem ts)'][mi], uid[mi]))}, "
             f"khong anh {ci(score_block(yi[mu], OOF['L0 (lam sang + diem ts)'][mu], uid[mu]))}")
print("\nKET LUAN C10:", KL["C10"])

                                ROC toan dev PR toan dev           ROC co anh PR co anh        ROC khong anh PR khong anh
mo hinh                                                                                                                  
L0 (lam sang + diem ts)  0.661 [0.640–0.681]       0.211  0.673 [0.643–0.703]     0.420  0.641 [0.615–0.667]        0.130
L0-tho (chi bang tho)    0.663 [0.642–0.683]       0.210  0.675 [0.646–0.705]     0.422  0.644 [0.619–0.670]        0.130
L2                       0.729 [0.708–0.747]       0.338  0.754 [0.728–0.780]     0.516  0.651 [0.625–0.677]        0.140

KET LUAN C10: L0: co anh 0.673 [0.643–0.703], khong anh 0.641 [0.615–0.667]


## C11. So sánh ghép cặp L3b vs L2 và L0 vs L2 ở nhóm không ảnh

Bootstrap ghép cặp theo bệnh nhân trên cùng 5.311 người. KTC cắt qua 0 = chưa phân biệt được.

In [16]:
rows = []
for a, b in (("L3b", "L2"), ("L0 (lam sang + diem ts)", "L2"), ("L0-tho (chi bang tho)", "L2"),
             ("L3b (tai hien loi cu)", "L3b")):
    g_, lo, hi, p = pgap(OOF[a], OOF[b], mu)
    rows.append({"so sanh": f"{a}  -  {b}", "chenh ROC": round(g_, 4), "KTC 95%": f"[{lo:+.4f}; {hi:+.4f}]",
                 "P(a>b)": f"{p:.0%}", "phan biet duoc": "CO" if (lo > 0 or hi < 0) else "khong"})
t11 = pd.DataFrame(rows).set_index("so sanh"); print(t11)
KL["C11"] = "; ".join(f"{k}: {v['chenh ROC']:+.4f} {v['KTC 95%']}" for k, v in t11.head(2).iterrows())
print("\nKET LUAN C11:", KL["C11"])

                                chenh ROC             KTC 95% P(a>b) phan biet duoc
so sanh                                                                            
L3b  -  L2                         0.0008  [-0.0034; +0.0049]    66%          khong
L0 (lam sang + diem ts)  -  L2    -0.0104  [-0.0225; +0.0026]     6%          khong
L0-tho (chi bang tho)  -  L2      -0.0065  [-0.0187; +0.0065]    16%          khong
L3b (tai hien loi cu)  -  L3b      0.0013  [+0.0004; +0.0022]   100%             CO

KET LUAN C11: L3b  -  L2: +0.0008 [-0.0034; +0.0049]; L0 (lam sang + diem ts)  -  L2: -0.0104 [-0.0225; +0.0026]


## C12. Ablation B3b: thay điểm ảnh hàng xóm bằng (i) trung bình nhãn hàng xóm, (ii) điểm từ mô hình chỉ lâm sàng

Giữ nguyên mạng nơ-ron, fold, seed và **cùng tập hàng xóm có ảnh** như B3b; chỉ đổi thứ được lấy trung bình:
- **(i)** nhãn thật của hàng xóm — chỉ hàng xóm thuộc fold train, hàng xóm ở fold validation/test coi như vắng;
- **(ii)** điểm của hồi quy logistic **chỉ lâm sàng**, fit trên fold train.

Nếu (ii) ngang B3b thì phần lợi của B3b không cần đến ảnh. Bổ sung bản hồi quy logistic của cùng ablation.

In [17]:
t12 = bang(["B0", "B3b", "B3b-(i) nhan hang xom", "B3b-(ii) diem lam sang hang xom",
            "L3b", "L3b-(i) nhan hang xom", "L3b-(ii) diem lam sang hang xom"])
print(t12)
rows = []
for a, b in (("B3b", "B0"), ("B3b-(i) nhan hang xom", "B3b"), ("B3b-(ii) diem lam sang hang xom", "B3b"),
             ("B3b-(i) nhan hang xom", "B0"), ("B3b-(ii) diem lam sang hang xom", "B0"),
             ("L3b-(i) nhan hang xom", "L3b"), ("L3b-(ii) diem lam sang hang xom", "L3b")):
    g_, lo, hi, p = pgap(OOF[a], OOF[b], mu)
    rows.append({"so sanh (nhom khong anh)": f"{a}  -  {b}", "chenh ROC": round(g_, 4),
                 "KTC 95%": f"[{lo:+.4f}; {hi:+.4f}]", "P(a>b)": f"{p:.0%}",
                 "phan biet duoc": "CO" if (lo > 0 or hi < 0) else "khong"})
t12b = pd.DataFrame(rows).set_index("so sanh (nhom khong anh)"); print("\n", t12b)
KL["C12"] = "; ".join(f"{k}: {v['chenh ROC']:+.4f} {v['KTC 95%']}" for k, v in t12b.head(3).iterrows())
print("\nKET LUAN C12:", KL["C12"])

                                        ROC toan dev PR toan dev           ROC co anh PR co anh        ROC khong anh PR khong anh
mo hinh                                                                                                                          
B0                               0.633 [0.613–0.652]       0.190  0.645 [0.615–0.676]     0.393  0.618 [0.593–0.645]        0.115
B3b                              0.713 [0.691–0.733]       0.320  0.741 [0.716–0.768]     0.506  0.633 [0.605–0.658]        0.125
B3b-(i) nhan hang xom            0.716 [0.696–0.735]       0.316  0.741 [0.716–0.768]     0.499  0.636 [0.608–0.661]        0.127
B3b-(ii) diem lam sang hang xom  0.714 [0.693–0.733]       0.315  0.742 [0.716–0.769]     0.504  0.634 [0.607–0.659]        0.124
L3b                              0.729 [0.708–0.748]       0.337  0.753 [0.726–0.779]     0.514  0.652 [0.626–0.677]        0.140
L3b-(i) nhan hang xom            0.731 [0.710–0.749]       0.338  0.754 [0.728–0.781]     

## C13. Calibration (Brier, ECE) cho các mô hình chính

**Brier** = sai số bình phương trung bình giữa xác suất dự đoán và nhãn (càng thấp càng tốt).
**ECE** = độ lệch trung bình giữa xác suất dự đoán và tỷ lệ dương thực tế, 10 bin đều.
Mốc: mô hình tầm thường luôn đoán bằng tỷ lệ nền. Lưu ý: mọi mô hình ở đây huấn luyện với trọng số cân bằng
lớp nên xác suất bị đẩy lên có hệ thống — ECE lớn là hệ quả thiết kế, không phải lỗi tính toán.

In [18]:
def ece(yv, p, bins=10):
    e = np.linspace(0, 1, bins + 1); b_ = np.clip(np.digitize(p, e[1:-1]), 0, bins - 1); tot = 0.0
    for i in range(bins):
        m = b_ == i
        if m.any(): tot += m.mean() * abs(p[m].mean() - yv[m].mean())
    return tot
rows = []
for nm in ("L0 (lam sang + diem ts)", "L2", "L3b", "B0", "B2", "B3b", "GAT"):
    row = {"mo hinh": nm}
    for g_, m in (("co anh", mi), ("khong anh", mu)):
        p = OOF[nm][m]; yv = yi[m]
        row[f"Brier {g_}"] = round(float(np.mean((p - yv) ** 2)), 4); row[f"ECE {g_}"] = round(ece(yv, p), 4)
        row[f"p TB {g_}"] = round(float(p.mean()), 3)
    rows.append(row)
base = {"mo hinh": "(moc) doan ty le nen"}
for g_, m in (("co anh", mi), ("khong anh", mu)):
    pr = yi[m].mean(); base[f"Brier {g_}"] = round(pr * (1 - pr), 4); base[f"ECE {g_}"] = 0.0; base[f"p TB {g_}"] = round(pr, 3)
rows.append(base)
t13 = pd.DataFrame(rows).set_index("mo hinh"); print(t13)
tot = t13.drop(index="(moc) doan ty le nen")
KL["C13"] = (f"Brier khong anh tot nhat {tot['Brier khong anh'].min():.4f} ({tot['Brier khong anh'].idxmin()}) vs moc "
             f"{base['Brier khong anh']:.4f}; ECE khong anh {tot['ECE khong anh'].min():.3f}-{tot['ECE khong anh'].max():.3f}. "
             f"Xac suat trung binh cao hon ty le nen that -> can hieu chinh neu dung xac suat truc tiep")
print("\nKET LUAN C13:", KL["C13"])

                         Brier co anh  ECE co anh  p TB co anh  Brier khong anh  ECE khong anh  p TB khong anh
mo hinh                                                                                                       
L0 (lam sang + diem ts)        0.2263      0.2187        0.466           0.2181         0.3534           0.434
L2                             0.2562      0.2711        0.521           0.1598         0.2651           0.346
L3b                            0.2559      0.2697        0.520           0.1595         0.2637           0.345
B0                             0.2158      0.1728        0.381           0.1868         0.2736           0.355
B2                             0.2371      0.2304        0.457           0.1403         0.1797           0.253
B3b                            0.2370      0.2275        0.452           0.1368         0.1744           0.248
GAT                            0.2888      0.3375        0.588           0.1445         0.2332           0.314
(

---
# TỔNG HỢP 13 ĐẦU MỤC

In [19]:
ten = {"A1": "Lo/nguon anh", "A2": "Trung thu muc cu 535", "A3": "Shortcut metadata", "A4": "Don vi chia fold",
       "B5": "Embedding control/frozen", "B6": "Diem hang xom", "B7": "Node/nhan test", "B8": "Chu burn-in",
       "B9": "Nhan bnn & bien ts", "C10": "L0", "C11": "Ghep cap L3b/L0 vs L2", "C12": "Ablation B3b",
       "C13": "Calibration"}
for k in ten:
    print(f"[{k}] {ten[k]}\n     {KL.get(k, 'CHUA CO')}\n")

[A1] Lo/nguon anh
     18 lo (ke ca 'unknown'); ty le duong theo lo tu 5.5% den 100.0%; 1 lo toan am/toan duong; 15/17 lo chi thuoc 1 tinh; thu muc normal/silicosis khop ketqua 100.0%; 207 anh co nam trong ten file khac nam kham

[A2] Trung thu muc cu 535
     462/462 anh duong trung thu muc cu theo diem anh (462 trung SHA); 0 anh am trung; 73/535 anh cu KHONG co trong 1835

[A3] Shortcut metadata
     Metadata KHONG ten thu muc: ROC-AUC tot nhat 0.698 (LR 0.698 [0.670–0.725], GBM 0.674 [0.646–0.702]); nhom manh nhat: lo chup (0.704). Co ten thu muc: 1.000 (tam thuong vi thu muc = nhan)

[A4] Don vi chia fold
     Gom theo NGUOI (patient_uid), khong theo luot: 8030 luot = 7024 nguoi; 0 nguoi vat fold, 0 nguoi vat dev/test; cung mot nguoi = trung SDT, hoac thieu SDT thi trung ca 6 truong

[B5] Embedding control/frozen
     Ca hai: BioViL-T ResNet50, anh xam Resize 512 -> CenterCrop 448 -> 3 kenh, lay 256/512 chieu song. control = fine-tune tren SetA (bo du lieu NGOAI, 2129 anh, nhan bnn

---
# PHỤ LỤC — mã nguồn các hàm tái sử dụng (để toàn bộ script nằm trong file này)

In [20]:
import fusion.baselines as FB, imagefeat.build_index as BI
for f in (FB.standardise, FB.neighbour_score_mean, FB.image_probe_scores, FB.Net, FB.train_predict,
          FB.score_block, FB.boot_ci, FB.paired_gap, BI.extract_id, BI.extract_batch_date, PI.build_identity):
    print("#" * 100); print(inspect.getsource(f))

####################################################################################################
def standardise(train_rows: np.ndarray, X: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Chuan hoa bang trung binh/do lech chuan cua rieng hang TRAIN HOP LE."""
    sub = X[train_rows & valid]
    mu, sd = sub.mean(0), sub.std(0)
    sd[sd < 1e-8] = 1.0
    out = np.zeros_like(X)
    out[valid] = (X[valid] - mu) / sd
    return out

####################################################################################################
def neighbour_score_mean(edge_index: np.ndarray, scores: np.ndarray,
                         has_image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Trung binh DIEM NGUY CO cua cac hang xom co anh (1 chieu) + so hang xom."""
    src = np.concatenate([edge_index[0], edge_index[1]])
    dst = np.concatenate([edge_index[1], edge_index[0]])
    keep = has_image[dst]
    src, dst = src[keep], dst[keep]
    total = np.zeros(len(scores), dtype=np.flo